# TeCo task 1 × ZuCo SR: EEG, PCA and VAE RSA

**Choose T4 GPU → Run all. Use Zyion for both Google account prompts.**

One run compares three matrices: TeCo–TeCo, ZuCo–ZuCo and TeCo–ZuCo. It reuses
saved features: no EEG extraction, raw-file downloads or NLP annotation. New
Drive output has a **50 MB total cap**, including resumable models and figures.

Targets: positive/negative, neutral/nonneutral, noun/verb/adjective,
content/function and short/long recorded first fixation. Missing auxiliary
labels or insufficient exact40 groups are explicitly listed, not replaced.

**Outputs are RSA correlations, not classifier balanced accuracy.**


In [ ]:
# Check GPU before mounting Drive or installing anything.
import os, subprocess
os.environ['MPLBACKEND']='Agg'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError('Select T4 GPU, not TPU, then Run all.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU required: Runtime → Change runtime type → T4 GPU → Run all. No setup or analysis started.')
print('GPU:',torch.cuda.get_device_name(0),'| existing Colab torch:',torch.__version__,flush=True)


## What runs

- Three reader-heldout folds, plus a separate **all-data descriptive** analysis.
- Four inputs: 48 regional/time bins over 0–800 ms; **200–400 ms** bins;
  18 regional summaries; normalized 48 bins. The short window is **not** the
  paper's exact 224–304 ms window, which these 100 ms caches cannot recover.
- Original features, PCA, VAE embeddings averaged after encoding, and encoding
  the average. One shared unsupervised encoder serves both datasets per fold/input.
- Exact40 averages, balanced classes within each contributing reader, three fixed
  regroupings. Readers without 40 trials in every class are reported as excluded.
- Pre-onset EEG, word-property RDM/partial-rank diagnostics and reader-block
  shuffled-label tests. Only the primary grouping receives inferential tests.

Held-out readers can share words/sentences with training. These are the original
shared feature caches, not the separate paper-regression-cleaned ZuCo cohort.
Dataset/reader/equipment and word-property effects remain possible explanations.


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import sys, json
from IPython.display import display, Markdown, Image
drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/cross_dataset_rsa_task1'
REPO=Path('/content/word-sentiment-eeg-cross-rsa')
CODE_REVISION='45f3be82a0d2700d3d236805036d2dbf6033c58a'
def command(args):
    print('SETUP:', ' '.join(map(str,args)),flush=True)
    result=subprocess.run(list(map(str,args)),text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.stdout:print(result.stdout,flush=True)
    if result.returncode:raise RuntimeError('Setup failed; see the full diagnostic above. No analysis started.')
def git(*args):command(['git','-c','credential.helper=','-c','http.extraheader=',*args])
if not REPO.exists():git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
elif subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
    raise RuntimeError('Temporary checkout has edits; preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code does not match the pinned revision.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
print('Code revision:',CODE_REVISION,flush=True)

# Cloud-only small environment; reuse Colab's existing CUDA PyTorch installation.
# Never installs a second PyTorch/CUDA copy or changes imported notebook libraries.
ENV=Path('/content/word-eeg-cross-rsa-python');PYTHON=ENV/'bin/python'
command([sys.executable,'-m','venv','--system-site-packages','--without-pip',ENV])
command([sys.executable,'-m','pip','--python',PYTHON,'install','--only-binary=:all:',
         '-r',REPO/'requirements-paper-classification.txt'])
command([PYTHON,'-c','import numpy,pandas,scipy,sklearn,matplotlib,torch; '
         'assert torch.cuda.is_available(), "GPU unavailable in scientific environment"; '
         'print("Scientific environment ready:",numpy.__version__,pandas.__version__,torch.__version__)'])

# Verify the account and reserve only the remaining portion of the 50 MB cap.
print('Checking Zyion Drive account and quota…',flush=True)
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();about=response.json();quota=about['storageQuota']
email=about.get('user',{}).get('emailAddress','unavailable')
print('Authenticated quota account:',email,flush=True)
if email!='zyionygvmutch39@gmail.com':raise RuntimeError('Choose Zyion for authentication and Drive mounting; it contains the saved caches.')
if 'limit' not in quota:raise RuntimeError('Drive quota unavailable; cannot verify free space.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES<max(0,50_000_000-used):raise RuntimeError('Reserve at most 50 MB for this workflow. No EEG processing started.')
print(f'Free Drive: {FREE_DRIVE_BYTES/1e9:.2f} GB | output cap: 50 MB | existing output: {used/1e6:.1f} MB',flush=True)
print('Results:',OUT,flush=True)


## Run and resume

The first stage reads **24 saved subject caches**. It then trains small models
and evaluates RSA. Logs report subjects, input/fold, VAE epochs and saved cases.
Rerunning reuses completed models/cases and resumes the unfinished model.
Do not run two copies into this same output folder.


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
run_with_progress([str(PYTHON),'-u',str(REPO/'scripts/cross_dataset_rsa.py'),
                   '--thesis-root',str(THESIS),'--free-drive-bytes',str(FREE_DRIVE_BYTES),'--device','cuda'],
                  OUT/'runtime_status.json',activity='Task1 TeCo–ZuCo EEG/PCA/VAE RSA: saved features only')


## Main results: held-out readers

**RSA rho** measures agreement with the ideal class matrix. Positive values
mean greater between-class dissimilarity; it is not accuracy or R².
The table combines all three folds, requiring each fold to be available.

`reader_conditional_p` uses whole-reader class-label swaps, not independent-cell
tests. `holm_p` controls familywise error; `bh_fdr_q` controls exploratory FDR
across all available primary held-out tests. These are conditional on the recorded
stimuli and the reader-exchangeability assumption; no unseen-word or language
population claim follows. Regroupings 1/2 and all-data results are descriptive.


In [ ]:
import hashlib
import pandas as pd
state=json.loads((OUT/'complete.json').read_text())
assert state['status']=='completed'
for name,sha in state['checksums'].items():
    if hashlib.sha256((OUT/name).read_bytes()).hexdigest()!=sha:raise RuntimeError('Saved output checksum failed: '+name)
print('Completed and verified:',state['finished_at_utc'],'|',state['scope'])
summary=pd.read_csv(OUT/'heldout_summary.csv')
for target,g in summary.groupby('target',sort=False):
    display(Markdown('### '+target))
    with pd.option_context('display.max_rows',100,'display.max_columns',20):display(g.round(4))
if (OUT/'overview.png').exists():display(Image(filename=str(OUT/'overview.png')))


## Measured matrices and ideal models

Top row: measured similarities. Bottom row: ideal dissimilarity, **0 same / 1
different class**. Rows/columns are class-sorted exact40 averages, not individual
words. Self-similarity diagonals are blank. Within-reader cells can be visible,
but only between-reader cells enter within-dataset statistics.

Show the primary full-input sentiment figures below. All other saved targets
and representations are in `plots/`; numeric matrices are in matching `.npz`
files. We do not combine coordinates from independently fitted fold encoders.


In [ ]:
for mode in ['reader_fold1','all_data_descriptive']:
    for representation in ['features','vae_average_embeddings','vae_encode_average']:
        path=OUT/'plots'/mode/'sentiment_binary'/f'{representation}.png'
        if path.exists():
            display(Markdown('### '+mode+' — '+representation))
            display(Image(filename=str(path)))
        else:print('Figure unavailable (see coverage):',mode,representation)
print('All saved figures:',OUT/'plots')


## Coverage, controls and VAE diagnostics

Exact40 never silently becomes Max40. Some readers/targets can lack a complete
group in every class. Actual memberships, exclusions and shared-source fractions
are saved. Word-property diagnostics compare RDMs and partial ranks; they are
**not** duplicated-reader non-EEG classification results.

Reconstruction alone does not establish meaningful embeddings. Inspect latent
variation and dataset/reader dominance, and compare EEG/PCA/VAE and pre-onset.


In [ ]:
counts=pd.read_csv(OUT/'class_counts.csv')
display(counts.groupby(['target','dataset','label'],as_index=False).eligible_trials.sum())
print('Missing auxiliary targets:',json.loads((OUT/'missing_targets.json').read_text()))
unavailable=pd.read_csv(OUT/'unavailable.csv')
display(unavailable if len(unavailable)<50 else unavailable.head(50))
display(pd.read_csv(OUT/'label_examples.csv'))
diagnostics=pd.DataFrame(json.loads((OUT/'vae_diagnostics.json').read_text()))
display(diagnostics.drop(columns=['latent_std','pca_variance_explained','dataset_scalar_scales'],errors='ignore').round(4))
details=pd.read_csv(OUT/'results.csv')
display(Markdown('### Controls and fixed-regrouping sensitivity: full regional/time input'))
columns=['mode','target','representation','regrouping','matrix','rsa_rho','same_minus_different',
         'partial_rho_word_properties','property_model_rho','eeg_property_rdm_rho',
         'evaluated_samples_left','evaluated_samples_right','readers_left','readers_right',
         'fraction_pairs_sharing_source_occurrence','split_average_reliability']
with pd.option_context('display.max_rows',150,'display.max_columns',30):
    display(details.loc[details.input.eq('full_0_800'),columns].round(4))
print('ALL results (every input/fold/target/regrouping):',OUT/'results.csv')
print('Primary corrected results:',OUT/'heldout_summary.csv')
print('Exact event groups and exclusions:',OUT/'groups')
print('No task2/3 data, supervised fine-tuning, semantic-grouping classification or dense scalp maps were run here.')


## Interpretation and next extension

Both datasets differ in readers, stimuli, equipment and preparation. Shared
structure is cross-dataset evidence; differences do not isolate language.
These caches cannot test absolute scalp-voltage comparability or recreate the
separate paper-cleaned classifier. A positive RSA effect can reflect word or
fixation properties. No representation is selected using final evaluation scores.

After TeCo tasks 2/3 are uploaded and labelled, repeat this workflow with
**ZuCo SR+NR+TSR versus TeCo tasks1+2+3**, retaining this task-1 reference.

[Full protocol and sources](https://github.com/parmisbathayan/word-sentiment-eeg/blob/main/docs/CROSS_DATASET_RSA_VAE.md)
